In [1]:
import mysql.connector
import pandas as pd
import warnings
from pandasql import sqldf
import featureSetup as fs
import numpy as np
warnings.filterwarnings('ignore')

dbCredentials = fs.getSecret()
cnx = mysql.connector.connect(
    user=dbCredentials['username'],
    password=dbCredentials['password'],
    host=dbCredentials['host'],
    database=dbCredentials['dbInstanceIdentifier']
)
cursor = cnx.cursor()

print('We connected with secrets manager!')

def convert_decimal_to_float(decimal_value):
    return float(decimal_value)

We connected with secrets manager!


In [2]:
# query and store fight, fighters info

query = """
SELECT *
from fighterHyperlinks
"""
cursor.execute(query)
result = cursor.fetchall()
column_names = [column[0] for column in cursor.description]
fightersTable = pd.DataFrame(result, columns=column_names)


query = """
SELECT fs.*, ehl.eventDate
from fightStats fs LEFT JOIN
eventHyperlinks ehl ON ehl.eventID = fs.eventID
"""
cursor.execute(query)
result = cursor.fetchall()
column_names = [column[0] for column in cursor.description]
fightsTable = pd.DataFrame(result, columns=column_names)

#Clean the data we pulled so that we can generate the engineered features

# Split time columns and find actual length of bout in minutes
fightsTable[['minutes', 'seconds']] = fightsTable['time'].str.split(':', expand=True)
fightsTable['minutes'] = fightsTable['minutes'].astype(int)
fightsTable['seconds'] = fightsTable['seconds'].astype(int)
fightsTable['round'] = fightsTable['round'].astype(int)
fightsTable['Minutes In Fight'] = ((fightsTable['round'] - 1) * 5) + fightsTable['minutes'] + fightsTable['seconds'] / 60
fightsTable.drop(['minutes', 'seconds'], axis=1, inplace=True)
fightsTable['eventDate'] = pd.to_datetime(fightsTable['eventDate'])

converters = {'Strikes_Landed_Per_Minute': convert_decimal_to_float, 'Strikes_Absorbed_Per_Minute' : convert_decimal_to_float,
              'Takedown_Average' : convert_decimal_to_float, 'Submission_Average' : convert_decimal_to_float}
for column, func in converters.items():
    fightersTable[column] = fightersTable[column].apply(func)
fightersTable['DOB'] = pd.to_datetime(fightersTable['DOB'])


In [3]:
# pass in fight, fighter data 
result_df = fs.setupFeatures(fightsTable, fightersTable)

In [16]:
result_df.columns

Index(['fightID', 'fighterA', 'fighterB', 'fighterIDA', 'fighterIDB',
       'averageControlTimeA', 'averageControlTimeB', 'winPercentageA',
       'winPercentageB', 'averageFightTimeA', 'averageFightTimeB', 'ageA',
       'ageB', 'finishRateA', 'finishRateB', 'strikeDifferentialA',
       'strikeDifferentialB', 'takedownDifferentialA', 'takedownDifferentialB',
       'weightClassA', 'weightClassB', 'winStreakA', 'winStreakB',
       'controlPercentageA', 'controlPercentageB', 'numberOfFightsA',
       'numberOfFightsB', 'eventID'],
      dtype='object')